# ML Image Classification Pipeline - Evaluation Notebook

This notebook demonstrates the complete Machine Learning pipeline including:
- Data Acquisition
- Data Preprocessing
- Model Creation and Training
- Model Evaluation with comprehensive metrics
- Model Testing and Prediction


In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

from preprocessing import ImagePreprocessor
from model import ImageClassifierModel
from prediction import ImagePredictor

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


## 1. Data Acquisition

First, we'll set up the data paths and check what data we have available.


In [ ]:
# Define data paths
DATA_DIR = '../data/train'
TEST_DIR = '../data/test'
MODELS_DIR = '../models'

# Check if directories exist
print(f"Training data directory exists: {os.path.exists(DATA_DIR)}")
print(f"Test data directory exists: {os.path.exists(TEST_DIR)}")
print(f"Models directory exists: {os.path.exists(MODELS_DIR)}")

# If data doesn't exist, create a sample structure
if not os.path.exists(DATA_DIR):
    print(f"\n⚠️  Training data directory not found at {DATA_DIR}")
    print("Please organize your training images in the following structure:")
    print("data/train/")
    print("  ├── class1/")
    print("  │   ├── img1.jpg")
    print("  │   └── img2.jpg")
    print("  ├── class2/")
    print("  │   ├── img1.jpg")
    print("  │   └── img2.jpg")
    print("  └── ...")
else:
    # List available classes
    classes = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"\n📁 Available classes: {classes}")
    for cls in classes:
        cls_path = os.path.join(DATA_DIR, cls)
        img_count = len([f for f in os.listdir(cls_path) 
                        if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        print(f"  - {cls}: {img_count} images")


## 2. Data Preprocessing

Load and preprocess images from the training directory.


In [ ]:
# Initialize preprocessor
preprocessor = ImagePreprocessor(target_size=(224, 224), normalize=True)

# Load images from training directory
print("🔄 Loading images from training directory...")
try:
    images, labels, class_names = preprocessor.load_images_from_directory(DATA_DIR)
    
    print(f"\n✅ Successfully loaded data:")
    print(f"   - Total images: {len(images)}")
    print(f"   - Number of classes: {len(class_names)}")
    print(f"   - Classes: {class_names}")
    print(f"   - Image shape: {images[0].shape}")
    
    # Visualize class distribution
    label_counts = pd.Series(labels).value_counts().sort_index()
    print(f"\n📊 Class distribution:")
    for cls, count in label_counts.items():
        print(f"   - {cls}: {count} images")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("\n📝 Note: You can create sample data or download a dataset.")
    print("For this demonstration, we'll proceed with the assumption that data exists.")


### 2.1. Visualize Sample Images


In [ ]:
# Visualize sample images from each class
def visualize_samples(images, labels, class_names, num_samples=3):
    """Display sample images from each class"""
    fig, axes = plt.subplots(len(class_names), num_samples, 
                            figsize=(15, 5 * len(class_names)))
    if len(class_names) == 1:
        axes = axes.reshape(1, -1)
    
    for i, class_name in enumerate(class_names):
        class_indices = np.where(labels == class_name)[0]
        sample_indices = np.random.choice(class_indices, 
                                        min(num_samples, len(class_indices)), 
                                        replace=False)
        
        for j, idx in enumerate(sample_indices):
            ax = axes[i, j] if len(class_names) > 1 else axes[j]
            ax.imshow(images[idx])
            ax.set_title(f"{class_name}", fontsize=12, fontweight='bold')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Only visualize if data is loaded
if 'images' in locals() and len(images) > 0:
    visualize_samples(images, labels, class_names, num_samples=3)
else:
    print("⚠️  No images loaded. Skipping visualization.")


### 2.2. Encode Labels and Split Data


In [ ]:
# Encode labels to integers
encoded_labels = preprocessor.encode_labels(labels)

print(f"Label encoding completed:")
print(f"   - Original labels: {labels[:5]}...")
print(f"   - Encoded labels: {encoded_labels[:5]}...")

# Split data into train, validation, and test sets
if 'images' in locals() and len(images) > 0:
    X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.prepare_data(
        images, encoded_labels, test_size=0.2, validation_size=0.1, random_state=42
    )
    
    print(f"\n📊 Data split:")
    print(f"   - Training set: {X_train.shape[0]} images")
    print(f"   - Validation set: {X_val.shape[0]} images")
    print(f"   - Test set: {X_test.shape[0]} images")
    print(f"   - Image shape: {X_train[0].shape}")
    
    # Visualize data split
    split_counts = {
        'Train': len(X_train),
        'Validation': len(X_val),
        'Test': len(X_test)
    }
    
    plt.figure(figsize=(8, 5))
    plt.bar(split_counts.keys(), split_counts.values(), color=['#667eea', '#764ba2', '#f093fb'])
    plt.title('Data Split Distribution', fontsize=14, fontweight='bold')
    plt.ylabel('Number of Images')
    plt.xlabel('Dataset')
    for k, v in split_counts.items():
        plt.text(k, v + max(split_counts.values())*0.01, str(v), 
                ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available for splitting.")


## 3. Model Creation

Create and build the image classification model using transfer learning.


In [ ]:
# Create model
if 'class_names' in locals() and len(class_names) > 0:
    num_classes = len(class_names)
    model = ImageClassifierModel(
        num_classes=num_classes,
        input_shape=(224, 224, 3),
        model_name='image_classifier'
    )
    
    # Set class names
    model.set_class_names(class_names)
    
    # Build model with transfer learning
    print("🏗️  Building model architecture...")
    model.build_model(use_transfer_learning=True, dropout_rate=0.5)
    
    # Display model summary
    print("\n📋 Model Architecture:")
    model.model.summary()
    
    # Visualize model architecture (requires graphviz)
    try:
        from tensorflow.keras.utils import plot_model
        plot_model(model.model, to_file='../models/model_architecture.png', 
                  show_shapes=True, show_layer_names=True)
        print("\n✅ Model architecture saved to models/model_architecture.png")
    except:
        print("\n⚠️  Could not save model architecture diagram (graphviz not installed)")
else:
    print("⚠️  Cannot create model without class information.")


## 4. Model Training

Train the model with data augmentation.


In [ ]:
# Get data generator for augmentation
if 'X_train' in locals():
    datagen = preprocessor.get_data_generator(augment=True)
    
    print("🚀 Starting model training...")
    print("   - Using data augmentation")
    print("   - Transfer learning with MobileNetV2")
    print("   - Early stopping enabled")
    
    # Train model
    history = model.train(
        X_train, y_train, X_val, y_val,
        epochs=50,  # Can be adjusted
        batch_size=32,
        data_generator=datagen
    )
    
    print("\n✅ Training completed!")
    
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss plot
    axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display final metrics
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    
    print(f"\n📊 Final Training Metrics:")
    print(f"   - Training Accuracy: {final_train_acc:.4f}")
    print(f"   - Validation Accuracy: {final_val_acc:.4f}")
    print(f"   - Training Loss: {final_train_loss:.4f}")
    print(f"   - Validation Loss: {final_val_loss:.4f}")
else:
    print("⚠️  Cannot train model without training data.")


## 5. Model Evaluation

Comprehensive evaluation of the model using multiple metrics.


In [ ]:
# Evaluate model on test set
if 'X_test' in locals() and model.model is not None:
    print("🔍 Evaluating model on test set...")
    
    eval_results = model.evaluate(X_test, y_test, class_names=class_names)
    
    print("\n" + "="*60)
    print("MODEL EVALUATION RESULTS")
    print("="*60)
    
    # Overall Accuracy
    print(f"\n✅ Overall Test Accuracy: {eval_results['accuracy']:.4f} ({eval_results['accuracy']*100:.2f}%)")
    
    # Classification Report
    print("\n📊 Classification Report:")
    report_df = pd.DataFrame(eval_results['classification_report']).transpose()
    print(report_df.to_string())
    
    # Confusion Matrix
    cm = np.array(eval_results['confusion_matrix'])
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Per-class metrics visualization
    per_class_metrics = {
        'Precision': [eval_results['classification_report'][cls]['precision'] 
                     for cls in class_names],
        'Recall': [eval_results['classification_report'][cls]['recall'] 
                  for cls in class_names],
        'F1-Score': [eval_results['classification_report'][cls]['f1-score'] 
                    for cls in class_names]
    }
    
    metrics_df = pd.DataFrame(per_class_metrics, index=class_names)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    metrics_df.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.1])
    plt.tight_layout()
    plt.show()
    
    print(f"\n📈 Per-Class Metrics Summary:")
    print(metrics_df.to_string())
    
else:
    print("⚠️  Cannot evaluate model without test data or trained model.")


### 5.1. Additional Evaluation Metrics

Calculate additional metrics to assess model quality.


In [ ]:
# Additional metrics
if 'eval_results' in locals():
    from sklearn.metrics import precision_recall_curve, roc_curve, auc
    from sklearn.preprocessing import label_binarize
    
    # For multi-class, calculate macro and weighted averages
    macro_precision = eval_results['classification_report']['macro avg']['precision']
    macro_recall = eval_results['classification_report']['macro avg']['recall']
    macro_f1 = eval_results['classification_report']['macro avg']['f1-score']
    
    weighted_precision = eval_results['classification_report']['weighted avg']['precision']
    weighted_recall = eval_results['classification_report']['weighted avg']['recall']
    weighted_f1 = eval_results['classification_report']['weighted avg']['f1-score']
    
    print("📊 Aggregated Metrics:")
    print(f"   - Macro Average Precision: {macro_precision:.4f}")
    print(f"   - Macro Average Recall: {macro_recall:.4f}")
    print(f"   - Macro Average F1-Score: {macro_f1:.4f}")
    print(f"\n   - Weighted Average Precision: {weighted_precision:.4f}")
    print(f"   - Weighted Average Recall: {weighted_recall:.4f}")
    print(f"   - Weighted Average F1-Score: {weighted_f1:.4f}")
    
    # Create summary visualization
    summary_metrics = {
        'Overall Accuracy': eval_results['accuracy'],
        'Macro F1': macro_f1,
        'Weighted F1': weighted_f1,
        'Macro Precision': macro_precision,
        'Macro Recall': macro_recall
    }
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(summary_metrics.keys(), summary_metrics.values(), 
                   color=['#667eea', '#764ba2', '#f093fb', '#4facfe', '#00f2fe'])
    plt.title('Model Performance Summary', fontsize=14, fontweight='bold')
    plt.ylabel('Score', fontsize=12)
    plt.ylim([0, 1.1])
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()


## 6. Model Testing and Prediction

Test the model with new predictions and save the model.


In [ ]:
# Save the trained model
if model.model is not None:
    print("💾 Saving model...")
    model_path = model.save_model(model_dir=MODELS_DIR, save_format='tf')
    print(f"✅ Model saved to: {model_path}")
    
    # Load predictor
    predictor = ImagePredictor(model_path=model_path, class_names=class_names)
    print("✅ Predictor initialized")
    
    # Test prediction on a few test images
    if 'X_test' in locals() and len(X_test) > 0:
        print("\n🔮 Testing predictions on sample test images...")
        
        # Select random samples
        num_samples = min(5, len(X_test))
        sample_indices = np.random.choice(len(X_test), num_samples, replace=False)
        
        fig, axes = plt.subplots(1, num_samples, figsize=(20, 4))
        if num_samples == 1:
            axes = [axes]
        
        for i, idx in enumerate(sample_indices):
            test_image = X_test[idx]
            true_label_idx = y_test[idx]
            true_label = class_names[true_label_idx]
            
            # Make prediction (need to save image temporarily)
            import tempfile
            import cv2
            
            # Convert normalized image back for saving
            img_for_pred = (test_image * 255).astype(np.uint8)
            
            with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
                tmp_path = tmp.name
                cv2.imwrite(tmp_path, cv2.cvtColor(img_for_pred, cv2.COLOR_RGB2BGR))
                
                prediction = predictor.predict(tmp_path)
                
                # Clean up
                os.unlink(tmp_path)
            
            # Display
            axes[i].imshow(test_image)
            pred_class = prediction['predicted_class']
            confidence = prediction['confidence']
            color = 'green' if pred_class == true_label else 'red'
            axes[i].set_title(f"True: {true_label}\nPred: {pred_class}\nConf: {confidence:.2f}", 
                            fontsize=10, color=color, fontweight='bold')
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print("\n✅ Prediction testing completed!")
else:
    print("⚠️  No trained model to save.")


## 7. Model Quality Assessment

### Summary

Based on the evaluation metrics:

1. **Accuracy**: Overall classification accuracy indicates how well the model performs on the test set
2. **Precision**: Measures the model's ability to correctly identify positive cases
3. **Recall**: Measures the model's ability to find all positive cases
4. **F1-Score**: Harmonic mean of precision and recall, provides balanced metric
5. **Confusion Matrix**: Shows detailed breakdown of correct and incorrect predictions

### Model Performance Indicators:

- ✅ **Good Performance**: Accuracy > 0.85, Balanced precision/recall
- ⚠️ **Fair Performance**: Accuracy 0.70-0.85, Some class imbalance
- ❌ **Poor Performance**: Accuracy < 0.70, Significant misclassifications

### Recommendations:

1. If accuracy is low, consider:
   - Adding more training data
   - Data augmentation
   - Fine-tuning hyperparameters
   - Trying different architectures

2. If certain classes perform poorly:
   - Check class balance in training data
   - Add more samples for underperforming classes
   - Adjust class weights during training

3. If overfitting observed (large gap between train and validation accuracy):
   - Increase dropout rate
   - Add more regularization
   - Use early stopping
   - Reduce model complexity
